# Week 6 — Evaluation

**Focus:** How do dependence, search, and costs change the strength of forecast evidence?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week06/week6_demos.ipynb)

## What this notebook is about

A small forecasting edge can matter when it repeats across genuinely distinct opportunities, but repetition, costs, search, and changing regimes all affect how much evidence we really have. This notebook asks how to turn forecast performance into a well-supported conclusion.

The demonstrations move from cross-sectional rank information and decile spreads to breadth, gross-to-net performance, multiple testing and the deflated Sharpe ratio, changing model capacity, and time-varying states. Optional state-space and hidden-state examples show when a latent regime is recoverable and when the data do not identify it.

By the end, you should be able to:

- distinguish the number of rows from the number of meaningfully independent opportunities;
- connect rank information, breadth, turnover, and net performance;
- record the breadth of a model or strategy search;
- treat window length and model capacity as selected choices; and
- state what a latent-state estimate does not prove.

These are controlled experiments and sensitivity analyses. Their purpose is to calibrate evidence, not to advertise a strategy.

### Course context

For a concise review and the optional textbook route, see [Week 6 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week06/week6_summary.md) and [Week 6 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week06/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Per-period rank IC and IC-IR on a simulated cross-section — <a href="#Demo-1-%E2%80%94-Per-period-rank-IC-and-IC-IR-on-a-simulated-cross-section">Jupyter</a> · <a href="#scrollTo=md-866d627c647b">Colab</a></li>
<li>Demo 2 — Monotonic decile spread — <a href="#Demo-2-%E2%80%94-Monotonic-decile-spread">Jupyter</a> · <a href="#scrollTo=md-80799bca3b2b">Colab</a></li>
<li>Real-data companion — industry momentum rank IC, spread, and turnover — <a href="#Real-data-companion-%E2%80%94-industry-momentum-rank-IC,-spread,-and-turnover">Jupyter</a> · <a href="#scrollTo=md-bfd230f93c6a">Colab</a></li>
<li>Demo 3 — Fundamental law: IR ≈ IC × sqrt(breadth) — <a href="#Demo-3-%E2%80%94-Fundamental-law%3A-IR-%E2%89%88-IC-%C3%97-sqrt(breadth)">Jupyter</a> · <a href="#scrollTo=md-e5b865c586f1">Colab</a></li>
<li>Demo 4 — Gross-to-net Sharpe under transaction costs — <a href="#Demo-4-%E2%80%94-Gross-to-net-Sharpe-under-transaction-costs">Jupyter</a> · <a href="#scrollTo=md-8ef3a0ad8aa6">Colab</a></li>
<li>Demo 5 — Multiple-testing mirage — <a href="#Demo-5-%E2%80%94-Multiple-testing-mirage">Jupyter</a> · <a href="#scrollTo=md-6d40683d61aa">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 6 — Move the interpolation peak — <a href="#Demo-6-%E2%80%94-Move-the-interpolation-peak">Jupyter</a> · <a href="#scrollTo=md-cacb0ce99b2c">Colab</a></li>
<li>Demo 7 — Sweep one training-window dimension (optional extension) — <a href="#Demo-7-%E2%80%94-Sweep-one-training-window-dimension-(optional-extension)">Jupyter</a> · <a href="#scrollTo=md-80dabfe165e4">Colab</a></li>
</ul>

### Optional frontier

<ul>
<li>Demo 8 — Kalman filter for time-varying beta — <a href="#Demo-8-%E2%80%94-Kalman-filter-for-time-varying-beta">Jupyter</a> · <a href="#scrollTo=md-cfaa9ca476a3">Colab</a></li>
<li>Demo 9 — A 2-state Gaussian-emission HMM (recoverable case) — <a href="#Demo-9-%E2%80%94-A-2-state-Gaussian-emission-HMM-(recoverable-case)">Jupyter</a> · <a href="#scrollTo=md-ee9849063e29">Colab</a></li>
<li>Demo 10 — §7 knob-dial: when is a regime recoverable? — <a href="#Demo-10-%E2%80%94-%C2%A77-knob-dial%3A-when-is-a-regime-recoverable%3F">Jupyter</a> · <a href="#scrollTo=md-5a557801a8b6">Colab</a></li>
</ul>

## Project research-record checkpoint

This is the first project-phase notebook. Copy `project/RESEARCH_RECORD_TEMPLATE.md` to the root of your own project repository, name it exactly `RESEARCH_RECORD.md`, and commit the initialized file. **Do not copy the project record into this notebook.** Use this week's exercises to make these project fields concrete where applicable:

1. **Identity:** question, intended decision, owner, repository, and current commit.
2. **Information contract:** decision time, feature cutoff, outcome interval, data source and vintage, sample dates, and units.
3. **Baseline and test:** feasible baseline, time-aware development/selection/final-test roles, primary metric, and denominator.
4. **Search:** candidates tried or planned, breadth or dependence assumptions, and the rule for counting trials.
5. **Claim boundary:** costs or frictions, one important limitation, and the narrow claim the design could warrant.

### Three recorded-lecture breaks and one optional extension

1. **Rank and diagnose — Demos 1–2:** compute per-date rank IC and a decile spread from the same scores; separate ordering evidence from a net decision result.
2. **Scale, implement, and search — Demos 3–5:** calibrate the independent-breadth approximation, carry one score through two cost ledgers, calculate a search-adjusted Sharpe statistic, and test the development-selected noise strategy later.
3. **Interpolate — Demo 6:** vary a declared control and distinguish a second descent from evidence that the high-capacity endpoint is best.
4. **Optional stress-memory extension — Demo 7:** hold outer dates fixed while window length changes; compare the window selected by predictive loss with the one selected by turnover.

Demos 8–10 accompany the optional state-space and HMM frontier reading; they are not additional recorded-lecture breaks.

Append a dated decision-log entry whenever this week's evidence changes the project design. Mark fields `not applicable yet` rather than inventing them.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, norm

rng = np.random.default_rng(42)


def annualized_sharpe(values, periods_per_year=252):
    """Annualized mean-to-volatility ratio under a declared frequency."""
    values = np.asarray(values, dtype=float)
    return values.mean() / values.std(ddof=1) * np.sqrt(periods_per_year)


def ewma(values, alpha):
    """A small explicit EWMA helper used by the implementation experiment."""
    result = np.empty_like(values, dtype=float)
    result[0] = values[0]
    for index in range(1, len(values)):
        result[index] = alpha * values[index] + (1 - alpha) * result[index - 1]
    return result


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url

### Demo 1 — Per-period rank IC and IC-IR on a simulated cross-section

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
T, N = 252, 200          # 1 yr daily, 200 assets
latent_alignment = 0.05  # correlation with the latent return component
demo1_rng = np.random.default_rng(601)
# Generate target as a noisy view of a true score
true = demo1_rng.standard_normal((T, N))
noise = demo1_rng.standard_normal((T, N))
# Mix to control IC: pred = a*true + b*noise
a = latent_alignment; b = np.sqrt(1 - a**2)
pred = a * true + b * noise
ret = true * 0.01 + demo1_rng.standard_normal((T, N)) * 0.01  # actual realized

ics = np.array([spearmanr(pred[t], ret[t]).statistic for t in range(T)])
mean_ic = ics.mean()
ic_ir = mean_ic / ics.std() * np.sqrt(252)
print(f"Mean rank IC: {mean_ic:+.4f}")
print(f"Annualized IC-IR: {ic_ir:.2f}")

plt.figure(figsize=(8, 3))
plt.bar(np.arange(T), ics, color=["tab:green" if v > 0 else "tab:red" for v in ics],
        alpha=0.5)
plt.axhline(mean_ic, color="k", lw=1)
plt.axhline(mean_ic + ics.std(), color="k", ls=":", lw=0.7)
plt.axhline(mean_ic - ics.std(), color="k", ls=":", lw=0.7)
plt.title(f"Per-period rank IC (mean={mean_ic:+.3f}, IC-IR≈{ic_ir:.1f})")
plt.tight_layout(); plt.show()
# Expected: mean rank IC ~0.03. The annualized IC-IR looks implausibly high because the
#           simulation has no regime variation; real IC-IRs are far lower.

#### After you run

Record the mean rank IC, the annualized IC-IR, and the unit at which IC was
sampled. Then explain why this deliberately stable simulation can produce an
IC-IR that would be implausibly strong as an empirical market conclusion.

**Your notes:** `[write here]`

### Demo 2 — Monotonic decile spread

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
decile_returns = np.zeros((T, 10))
for t in range(T):
    order = np.argsort(pred[t])
    bins = np.array_split(order, 10)
    for d in range(10):
        decile_returns[t, d] = ret[t, bins[d]].mean()
avg_per_decile = decile_returns.mean(axis=0)

plt.figure(figsize=(8, 3.5))
plt.bar(np.arange(10) + 1, avg_per_decile)
plt.xlabel("prediction decile (1=low, 10=high)")
plt.ylabel("mean realized return")
plt.title("Monotonic decile spread — the picture of a real signal")
plt.tight_layout(); plt.show()

long_short = decile_returns[:, 9] - decile_returns[:, 0]
sharpe_gross = long_short.mean() / long_short.std() * np.sqrt(252)
print(f"Top-minus-bottom gross Sharpe (annualized): {sharpe_gross:.2f}")
print("Reminder: GROSS. Costs and turnover come next.")

#### After you run

Describe the shape across all ten deciles and record the gross top-minus-bottom
Sharpe. State what this diagnostic adds beyond IC and what it still omits before
the spread could be described as a net investment result.

**Your notes:** `[write here]`

### Real-data companion — industry momentum rank IC, spread, and turnover

> **Optional empirical comparison:** Deck A, after recording segment S1. Run this after the known-truth demo. Use the same transformation and compare the simulation with the historical series. Record one important difference and the most likely reason for it.

In [ ]:
industry_raw, industry_source = course_csv("datasets/famafrench/ff_12industry_daily.csv")
industry_raw["date"] = pd.to_datetime(industry_raw["date"].astype(str), format="%Y%m%d")
industry_returns = industry_raw.set_index("date").sort_index().loc["1990":] / 100.0
industry_score = industry_returns.rolling(63).sum()
industry_next = industry_returns.shift(-1)
industry_mask = industry_score.notna().all(axis=1) & industry_next.notna().all(axis=1)
industry_score = industry_score.loc[industry_mask]
industry_next = industry_next.loc[industry_mask]

industry_ic = pd.Series(
    [spearmanr(industry_score.loc[date], industry_next.loc[date]).statistic
     for date in industry_score.index],
    index=industry_score.index,
)
industry_weights = np.zeros_like(industry_score.to_numpy())
for row, values in enumerate(industry_score.to_numpy()):
    order = np.argsort(values)
    industry_weights[row, order[:3]] = -1 / 3
    industry_weights[row, order[-3:]] = 1 / 3
industry_gross = np.sum(industry_weights * industry_next.to_numpy(), axis=1)
industry_turnover = np.r_[np.abs(industry_weights[0]).sum(),
                          np.abs(np.diff(industry_weights, axis=0)).sum(axis=1)]
industry_net = industry_gross - 5e-4 * industry_turnover

print("Source: Kenneth R. French Data Library, bundled 12 industry portfolios")
print(f"File: {industry_source}")
print(f"Sample: {industry_score.index.min().date()} through {industry_score.index.max().date()}")
print("Units: decimal daily returns; score is trailing 63-session return through t; target is t+1")
print(f"Mean daily cross-sectional rank IC: {industry_ic.mean():+.4f}")
print(f"Top-three minus bottom-three gross Sharpe: {annualized_sharpe(industry_gross):+.2f}")
print(f"Mean one-way turnover proxy: {industry_turnover.mean():.2f}")
print(f"Net Sharpe at 5 bps per unit turnover: {annualized_sharpe(industry_net):+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(industry_ic.rolling(252).mean(), lw=0.8)
axes[0].axhline(0, color="0.3", lw=0.5)
axes[0].set_title("One-year rolling mean rank IC")
axes[1].plot(industry_score.index, np.cumsum(industry_gross), label="gross")
axes[1].plot(industry_score.index, np.cumsum(industry_net), label="net of 5 bps")
axes[1].set_title("A real spread needs a cost ledger")
axes[1].legend()
fig.tight_layout(); plt.show()
print("Twelve industry portfolios are not twelve independent bets; the IC-IR would need dependence-aware inference.")

#### After you run

Record the source, sample dates, mean IC, gross spread Sharpe, turnover, and net
Sharpe. Compare the historical path with the known-truth simulation and give
one reason the empirical result is less stable.

**Your notes:** `[write here]`

### Demo 3 — Fundamental law: IR ≈ IC × sqrt(breadth)

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
ic_per_bet = 0.05
breadths = [1, 5, 25, 100, 400, 1600]
realized_irs, theoretical = [], []
demo3_rng = np.random.default_rng(603)
n_periods = 4000
for B in breadths:
    forecast = demo3_rng.standard_normal((n_periods, B))
    outcome = (ic_per_bet * forecast
               + np.sqrt(1 - ic_per_bet**2)
               * demo3_rng.standard_normal((n_periods, B)))
    active_return = np.mean(forecast * outcome, axis=1)
    realized_irs.append(active_return.mean() / active_return.std(ddof=1))
    theoretical.append(ic_per_bet * np.sqrt(B))

print("Breadth | realized IR | IC*sqrt(B)")
for B, r, t in zip(breadths, realized_irs, theoretical):
    print(f"  {B:5d}  | {r:9.3f}   | {t:7.3f}")
# Expected: the two columns roughly track.

#### After you run

Compare the simulated information ratios with `IC × sqrt(breadth)`. Name two
assumptions that would make nominal breadth overstate attainable performance in
a real financial application.

**Your notes:** `[write here]`

### Demo 4 — Gross-to-net Sharpe under transaction costs

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
demo4_rng = np.random.default_rng(604)
n_days = 3500
persistence = 0.96
state = np.empty(n_days)
state[0] = demo4_rng.standard_normal()
for day in range(1, n_days):
    state[day] = (persistence * state[day - 1]
                  + np.sqrt(1 - persistence**2) * demo4_rng.standard_normal())

score = state + 1.1 * demo4_rng.standard_normal(n_days)
next_return = 0.0005 * state + 0.01 * demo4_rng.standard_normal(n_days)
raw_weight = np.tanh(score)
implementations = {
    "slow": ewma(raw_weight, alpha=0.08),
    "fast": raw_weight,
}
cost_grid = np.linspace(0, 60, 61)
cost_rows, ledger_rows = [], []
for label, weight in implementations.items():
    weight = weight / np.sqrt(np.mean(weight**2))
    trade = np.r_[abs(weight[0]), np.abs(np.diff(weight))]
    gross = weight * next_return
    breakeven_bps = gross.mean() / trade.mean() * 10_000
    for cost_bps in cost_grid:
        net = gross - cost_bps / 10_000 * trade
        cost_rows.append({
            "implementation": label,
            "cost_bps": cost_bps,
            "net_sharpe": annualized_sharpe(net),
        })
    ledger_rows.append({
        "implementation": label,
        "gross_mean_bps": gross.mean() * 10_000,
        "gross_sharpe": annualized_sharpe(gross),
        "turnover": trade.mean(),
        "breakeven_bps": breakeven_bps,
        "net_sharpe_10bps": annualized_sharpe(gross - 10 / 10_000 * trade),
    })

cost_results = pd.DataFrame(cost_rows)
implementation_ledger = pd.DataFrame(ledger_rows).set_index("implementation")
print(implementation_ledger.round(3).to_string())

plt.figure(figsize=(7.5, 3.7))
for label, color in [("slow", "green"), ("fast", "red")]:
    rows = cost_results[cost_results.implementation == label]
    plt.plot(rows.cost_bps, rows.net_sharpe, color=color, label=label)
    plt.axvline(implementation_ledger.loc[label, "breakeven_bps"],
                color=color, ls=":", lw=1)
plt.axhline(0, color="k", lw=0.5)
plt.xlabel("cost per dollar traded (bps)")
plt.ylabel("net Sharpe (annualized)")
plt.title("One score, two implementations, explicit cost sensitivity")
plt.legend(); plt.tight_layout(); plt.show()
# Expected: the fast implementation has much greater turnover and a much lower
#           break-even cost even though both begin from the same score.

#### After you run

Identify which implementation loses performance faster as cost rises and
explain why. Record the approximate break-even cost from the plot and state
which quantities were held fixed during the cost sweep.

**Your notes:** `[write here]`

### Demo 5 — Multiple-testing mirage

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
T = 252 * 10
N_strats = 300
demo5_rng = np.random.default_rng(605)
development = demo5_rng.standard_normal((N_strats, T)) * 0.01
test = np.random.default_rng(655).standard_normal((N_strats, T)) * 0.01
development_sharpes = (development.mean(axis=1) / development.std(axis=1, ddof=1)
                       * np.sqrt(252))
test_sharpes = (test.mean(axis=1) / test.std(axis=1, ddof=1)
                      * np.sqrt(252))
selected = int(np.argmax(development_sharpes))
best_so_far = np.maximum.accumulate(development_sharpes)

# In this controlled experiment the 300 trials are independent and fully logged.
# Estimate the expected best null Sharpe, then place that benchmark inside
# the probabilistic-Sharpe approximation used by the deflated Sharpe ratio.
cross_trial_sigma = development_sharpes.std(ddof=1)
expected_best_null = cross_trial_sigma * (
    (1 - np.euler_gamma) * norm.ppf(1 - 1 / N_strats)
    + np.euler_gamma * norm.ppf(1 - 1 / (N_strats * np.e))
)
selected_returns = development[selected]
standardized = ((selected_returns - selected_returns.mean())
                / selected_returns.std(ddof=0))
sample_skew = np.mean(standardized ** 3)
sample_kurtosis = np.mean(standardized ** 4)  # non-excess kurtosis
selected_sr_period = development_sharpes[selected] / np.sqrt(252)
null_sr_period = expected_best_null / np.sqrt(252)
dsr_denominator = np.sqrt(
    1 - sample_skew * selected_sr_period
    + (sample_kurtosis - 1) / 4 * selected_sr_period ** 2
)
deflated_sharpe = norm.cdf(
    (selected_sr_period - null_sr_period) * np.sqrt(T - 1)
    / dsr_denominator
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(np.arange(1, N_strats + 1), best_so_far)
axes[0].axhline(expected_best_null, color="tab:orange", ls="--",
                label="expected best null")
axes[0].set_xlabel("strategies tried (cumulative)")
axes[0].set_ylabel("best development Sharpe")
axes[0].set_title("The selected noise result rises with search")
axes[0].legend()
axes[1].bar(["development", "later test"],
            [development_sharpes[selected], test_sharpes[selected]],
            color=["tab:red", "tab:green"])
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_ylabel("annualized Sharpe")
axes[1].set_title("The selected index on untouched noise")
plt.tight_layout(); plt.show()
print(f"Candidates searched: {N_strats}")
print(f"Selected development Sharpe: {development_sharpes[selected]:+.2f}")
print(f"Expected best null Sharpe: {expected_best_null:+.2f}")
print(f"Deflated Sharpe ratio: {deflated_sharpe:.2f}")
print(f"Same selected strategy, later Sharpe: {test_sharpes[selected]:+.2f}")
print("Every candidate has zero population skill by construction.")
# Expected: selection produces a positive development winner; its later result
#           is another draw around the zero planted truth.

#### After you run

Record the number of strategies searched, the selected development Sharpe, the
expected best null Sharpe, the deflated Sharpe ratio, and the same selected
index's later Sharpe. The true Sharpe of every candidate is zero. Explain what
the deflated statistic adds, why it depends on a complete search record, and why
the later value need not be exactly zero.

**Your notes:** `[write here]`

### Demo 6 — Move the interpolation peak

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Edit these three controls, then rerun the cell: n, noise_sd, and spectrum_decay.
n = 120
noise_sd = 0.7
spectrum_decay = 0.0       # 0 = isotropic; larger values create a steeper spectrum
p_grid = np.array([12, 30, 60, 90, 108, 116, 120, 124, 135, 170, 240, 360])
n_trials = 12
p_max = int(p_grid.max())

train_mse = np.zeros((n_trials, len(p_grid)))
test_mse = np.zeros_like(train_mse)
for trial in range(n_trials):
    g = np.random.default_rng(1000 + trial)
    scales = np.exp(-spectrum_decay * np.arange(p_max) / p_max)
    beta = np.zeros(p_max)
    beta[:8] = np.linspace(1.0, 0.25, 8) / np.sqrt(8)
    X_train_all = g.standard_normal((n, p_max)) * scales
    X_test_all = g.standard_normal((2000, p_max)) * scales
    y_train = X_train_all @ beta + noise_sd * g.standard_normal(n)
    y_test = X_test_all @ beta + noise_sd * g.standard_normal(2000)

    for j, p in enumerate(p_grid):
        X_train = X_train_all[:, :p]
        X_test = X_test_all[:, :p]
        # A tiny numerical ridge stabilizes the exact threshold without
        # changing the minimum-norm solution away from singularity.
        eps = 1e-10
        if p <= n:
            coef = np.linalg.solve(
                X_train.T @ X_train + eps * np.eye(p), X_train.T @ y_train
            )
        else:
            coef = X_train.T @ np.linalg.solve(
                X_train @ X_train.T + eps * np.eye(n), y_train
            )
        train_mse[trial, j] = np.mean((y_train - X_train @ coef) ** 2)
        test_mse[trial, j] = np.mean((y_test - X_test @ coef) ** 2)

mean_test = test_mse.mean(axis=0)
peak_ratio = p_grid[np.argmax(mean_test)] / n
print(f"Interpolation peak occurs near p/n = {peak_ratio:.2f}")
print("Change n, noise_sd, or spectrum_decay and see what moves.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(p_grid / n, train_mse.mean(axis=0), "o-", label="train MSE")
ax.plot(p_grid / n, mean_test, "s-", label="held-out MSE")
ax.axvline(1.0, color="0.25", ls="--", lw=1, label="p/n = 1")
ax.set_yscale("log")
ax.set_xlabel("parameter ratio p/n")
ax.set_ylabel("mean squared error (log scale)")
ax.set_title("Interpolation peak and second descent on held-out data")
ax.legend()
plt.tight_layout(); plt.show()

#### After you run

Locate the interpolation threshold on the feature-count axis and describe test
error on both sides of it. Change one declared input—sample size, noise, or
feature count—and state your directional prediction before rerunning.

**Your notes:** `[write here]`

### Demo 7 — Sweep one training-window dimension (optional extension)

> **Optional extension:** This is not a required recorded-lecture break. Predict what the output should show, then run the cell. Record the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Every candidate faces the same outer dates, features, objective, and position rule.
g = np.random.default_rng(2026)
T, K = 900, 12
X = g.standard_normal((T, K))
beta_t = np.zeros((T, K))
beta_t[:, :3] = np.column_stack([
    0.35 * np.sin(np.arange(T) / 95),
    0.25 * np.cos(np.arange(T) / 130),
    0.20 * np.sin(np.arange(T) / 170 + 0.8),
])
y = np.sum(X * beta_t, axis=1) + 0.65 * g.standard_normal(T)

windows = [30, 60, 120, 240]
first_outer = max(windows)
rows = []
loss_paths = {}
for window in windows:
    forecasts = []
    for t in range(first_outer, T):
        Xw, yw = X[t-window:t], y[t-window:t]
        coef = np.linalg.solve(Xw.T @ Xw + 5.0 * np.eye(K), Xw.T @ yw)
        forecasts.append(X[t] @ coef)
    forecasts = np.asarray(forecasts)
    outcomes = y[first_outer:]
    losses = (outcomes - forecasts) ** 2
    # A fixed bounded mapping keeps the decision comparison identical.
    positions = np.tanh(forecasts / 0.5)
    turnover = np.abs(np.diff(positions)).mean()
    loss_paths[window] = pd.Series(losses).rolling(40).mean().to_numpy()
    rows.append({"window": window, "MSE": losses.mean(), "turnover": turnover})

window_result = pd.DataFrame(rows).set_index("window")
print(window_result.round(3))
print("All rows use the same outer forecast dates:", first_outer, "through", T - 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for window in windows:
    axes[0].plot(loss_paths[window], lw=1, label=f"{window}")
axes[0].set_title("Rolling predictive loss")
axes[0].set_xlabel("common outer forecast")
axes[0].set_ylabel("40-period MSE")
axes[0].legend(title="window", ncol=2, fontsize=8)
axes[1].scatter(window_result["MSE"], window_result["turnover"], s=55)
for window, row in window_result.iterrows():
    axes[1].annotate(str(window), (row["MSE"], row["turnover"]), xytext=(4, 4),
                     textcoords="offset points")
axes[1].set_xlabel("predictive MSE")
axes[1].set_ylabel("mean turnover")
axes[1].set_title("One sweep, two ledgers")
plt.tight_layout(); plt.show()

#### After you run

Record the window selected by predictive loss and whether the turnover result
points to the same window. List the features, outer forecast dates, and other
settings that remained fixed so that the sweep has one interpretation.

**Your notes:** `[write here]`

### Demo 8 — Kalman filter for time-varying beta

> **Guided-reading demo:** Run this with the optional frontier slides. Treat the output as a mechanism check, not as an empirical market result.

In [ ]:
T = 1000
demo8_rng = np.random.default_rng(608)
true_beta = 0.5 + np.cumsum(demo8_rng.standard_normal(T) * 0.005)
mkt_ret = demo8_rng.standard_normal(T) * 0.01
stock_ret = true_beta * mkt_ret + demo8_rng.standard_normal(T) * 0.005

# Kalman: scalar state beta_t = beta_{t-1} + w, w ~ N(0, q)
#         obs:   stock_ret = beta_t * mkt_ret + v, v ~ N(0, r)
q = 1e-5
r = 0.005 ** 2
beta_hat = np.zeros(T); P = np.zeros(T)
beta_hat[0] = 0.5; P[0] = 0.5
for t in range(1, T):
    # predict
    beta_pred = beta_hat[t-1]
    P_pred = P[t-1] + q
    # update
    H = mkt_ret[t]
    S = H * P_pred * H + r
    K = P_pred * H / S
    beta_hat[t] = beta_pred + K * (stock_ret[t] - H * beta_pred)
    P[t] = (1 - K * H) * P_pred

# Compare to a 60-day rolling OLS
roll_beta = np.zeros(T)
W = 60
for t in range(W, T):
    x = mkt_ret[t-W:t]; y = stock_ret[t-W:t]
    roll_beta[t] = (x @ y) / (x @ x)
roll_beta[:W] = np.nan

plt.figure(figsize=(8, 3.5))
plt.plot(true_beta, color="k", ls="--", label="true beta", lw=1)
plt.plot(roll_beta, color="gray", label="60-day rolling OLS", lw=0.8)
plt.plot(beta_hat, color="red", label="Kalman", lw=1)
plt.legend(); plt.xlabel("t"); plt.ylabel("beta")
plt.title("Time-varying beta: filter vs. rolling OLS")
plt.tight_layout(); plt.show()
# Expected: the filtered beta tracks the moving true beta with a lag set by the ratio of
#           process to measurement variance.

#### After you run

Compare the filtered estimate with the planted beta path. Describe the tradeoff
created by the process-variance setting and state why the filtered state remains
an estimate rather than an observed regime.

**Your notes:** `[write here]`

### Demo 9 — A 2-state Gaussian-emission HMM (recoverable case)

> **Guided-reading demo:** Run this with the optional frontier slides. Treat the output as a mechanism check, not as an empirical market result.

In [ ]:
# Two regimes: low-vol (sigma=1%), high-vol (sigma=3%).
# Self-transition probability 0.98.
T = 2000
A = np.array([[0.98, 0.02], [0.02, 0.98]])
sig = np.array([0.01, 0.03])
mu = np.array([0.0, 0.0])
states = np.zeros(T, dtype=int)
y = np.zeros(T)
demo9_rng = np.random.default_rng(609)
for t in range(1, T):
    states[t] = demo9_rng.choice(2, p=A[states[t-1]])
y = mu[states] + sig[states] * demo9_rng.standard_normal(T)

# Forward filter (causal) for a known-parameter model
def hmm_filter(y, A, mu, sig, pi0=None):
    K = A.shape[0]; T = len(y)
    if pi0 is None: pi0 = np.ones(K) / K
    alpha = np.zeros((T, K))
    em = norm.pdf(y[:, None], loc=mu, scale=sig)
    alpha[0] = pi0 * em[0]; alpha[0] /= alpha[0].sum()
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * em[t]
        alpha[t] /= alpha[t].sum()
    return alpha

alpha = hmm_filter(y, A, mu, sig)
p_high_filtered = alpha[:, 1]

plt.figure(figsize=(8, 3.5))
plt.plot(p_high_filtered, color="orange", lw=0.7, label="filtered P(high-vol)")
plt.plot(states, color="k", lw=0.4, alpha=0.5, label="true state")
plt.legend(); plt.title("Filtered (causal) regime posterior — usable in real time")
plt.tight_layout(); plt.show()
# Expected: the fitted model recovers both states when they separate enough; which state
#           gets which label is arbitrary.

#### After you run

State which separation and persistence make the planted states recoverable in
this simulation. Record one reason a fitted state label should not be treated
as an observed economic fact.

**Your notes:** `[write here]`

### Demo 10 — §7 knob-dial: when is a regime recoverable?

> **Guided-reading demo:** Run this with the optional frontier slides. Treat the output as a mechanism check, not as an empirical market result.

In [ ]:
# Heatmap of OOS log-likelihood gain over a 1-state benchmark on a (Δ, p) grid.
deltas = np.linspace(0.0, 2.5, 6)
ps = np.linspace(0.6, 0.99, 6)
T = 1500
gain = np.zeros((len(ps), len(deltas)))
demo10_rng = np.random.default_rng(610)

for i, p in enumerate(ps):
    A_ = np.array([[p, 1-p], [1-p, p]])
    for j, d in enumerate(deltas):
        mu_ = np.array([-d/2, d/2])
        sig_ = np.array([1.0, 1.0])
        # simulate
        st = np.zeros(T, dtype=int)
        yt = np.zeros(T)
        for t in range(1, T):
            st[t] = demo10_rng.choice(2, p=A_[st[t-1]])
        yt = mu_[st] + sig_[st] * demo10_rng.standard_normal(T)
        ntr = T // 2
        # 1-state benchmark: fit mean and sd on training, score next half
        mu0 = yt[:ntr].mean(); sd0 = yt[:ntr].std()
        ll_bench = norm.logpdf(yt[ntr:], loc=mu0, scale=sd0).mean()
        # 2-state HMM with TRUE parameters (a charitable upper bound to EM):
        alpha_ = hmm_filter(yt[:ntr], A_, mu_, sig_)
        # one-step predictive density on the test block
        cur = alpha_[-1]
        ll_hmm = []
        for tt in range(ntr, T):
            pred = cur @ A_
            em_t = norm.pdf(yt[tt], loc=mu_, scale=sig_)
            ll_hmm.append(np.log((pred * em_t).sum() + 1e-300))
            post = pred * em_t
            cur = post / post.sum()
        gain[i, j] = np.mean(ll_hmm) - ll_bench

fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(gain, origin="lower", aspect="auto",
               extent=[deltas[0], deltas[-1], ps[0], ps[-1]],
               cmap="RdYlGn")
ax.set_xlabel(r"separation $\Delta$"); ax.set_ylabel("persistence p")
ax.set_title("OOS log-lik gain over 1-state benchmark")
plt.colorbar(im, ax=ax, label="gain (nats/obs)")
# zero-contour overlay
ax.contour(deltas, ps, gain, levels=[0.0], colors="black", linestyles="--")
plt.tight_layout(); plt.show()

print("Read the heatmap: bright region = HMM cleanly wins; dim = unrecoverable.")
print("The dashed contour is the zero-gain frontier — below it, an HMM buys nothing.")
# Expected: recoverability improves with separation and persistence. Below some
#           separation the states are simply not identified.

#### After you run

Use the heatmap to identify where state recovery improves. Explain separately
what greater emission separation and greater state persistence contribute, and
state why this known-truth result does not validate a market regime label.

**Your notes:** `[write here]`

## Where this leaves us

Forecast value depends on more than a headline metric: it depends on effective breadth, dependence, selection, turnover, costs, and stability across changing conditions. A regime label is also an estimated representation, not an observable fact.

Week 7 turns to the numerical and representational machinery behind optimization, dimension reduction, and clustering.